In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
import matplotlib.pyplot as plt
from IPython.display import Image, display
from datetime import datetime

import gc
import itertools

```python
"""
Paths:
Chunk0950                    /Chunk0950/001_007/AO2Dtree.root
Chunk1000                    [ /Chunk1000/001_005/AO2Dtree.root, /Chunk1000/006_009/AO2Dtree.root ]
Chunk1010                    [ /Chunk1010/001_005/AO2Dtree.root, /Chunk1010/006_010/AO2Dtree.root ]
Chunk1020                    /Chunk1020/001_007/AO2Dtree.root
Chunk1030                    [ /Chunk1030/001_005/AO2Dtree.root, /Chunk1030/006_011/AO2Dtree.root ]
Chunk1040                    /Chunk1040/001_007/AO2Dtree.root
Chunk1050                    [ /Chunk1050/001_005/AO2Dtree.root, /Chunk1050/006_010/AO2Dtree.root ]
Chunk1100                    [ /Chunk1100/001_005/AO2Dtree.root, /Chunk1100/006_009/AO2Dtree.root ]
Chunk1110                    /Chunk1110/001_005/AO2Dtree.root
Chunk1120                    [ /Chunk1120/001_005/AO2Dtree.root, /Chunk1120/006_011/AO2Dtree.root ]
Chunk1130                    /Chunk1130/001_008/AO2Dtree.root
Chunk1140                    [ /Chunk1140/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1150                    /Chunk1150/001_007/AO2Dtree.root
Chunk1200                    [ /Chunk1200/001_006/AO2Dtree.root, /Chunk1120/007_011/AO2Dtree.root ]
Chunk1210                    /Chunk1210/001_008/AO2Dtree.root
Chunk1220                    /Chunk1220/001_008/AO2Dtree.root
Chunk1300                    [ /Chunk1300/001_006/AO2Dtree.root, /Chunk1300/003/AO2Dtree.root, /Chunk1300/007_010/AO2Dtree.root ]
"""
```

In [2]:
#base_path = "/mnt/SingleTrackTrees/Data/alice_data_2023_LHC23f_535087_apass4_1300_Thinner/Output/Run535087"
base_path = "~/Downloads"
which_chunk = "Chunk0950"
which_number = "001_007"
which_file = "AO2Dtree.root"
path = base_path + "/".join(["/", which_chunk, which_number, which_file])
#path = "~/Downloads/AO2Dtree.root"
file = uproot.open(path)

In [3]:
# !!! NEW CUTS v2!!!

# Kaon hypothesis
Ka_hyp = " ( fNsigmaTPCka.abs() < 3 ) & "\
         " ( \
             ( (fPt <= 1.4) & (fNsigmaTOFka.abs() <=7) ) | \
             ( (fPt.between(1.4,3.0)) & (fNsigmaTOFka.abs() <= 4)  )| \
             ( (fPt < 3.0) & (fNsigmaTOFka == -999))\
            )"

# Pion Hypothesis
Pi_hyp = " ( fNsigmaTPCpi.abs() < 3 ) & "\
         " ( \
             ( (fPt <= 1.4) & (fNsigmaTOFpi.abs() <=7) ) | \
             ( (fPt.between(1.4,3.0)) & (fNsigmaTOFpi.abs() <= 4) ) | \
             ( (fPt < 3.0) & (fNsigmaTOFpi == -999))\
            )"

cut_expression = f"({Ka_hyp} | {Pi_hyp})"

out_name = f"Cuts_for_{which_chunk}_{which_number}.txt"
with open(out_name, "w") as f:
  f.write(cut_expression)

In [4]:
def to_global_coord( row ):
    xy_in = np.array([row["fX"],row["fY"]])
    # angle = -row["fAlpha"]
    # rot = np.array([[np.cos(angle),np.sin(angle)], [-np.sin(angle), np.cos(angle)]]) # inverse matrix of the one from to_track_coord (-alfa)
    # Rewriting the above matrix using cosine/sin even/odd function properties: less multiplications
    angle = row["fAlpha"]
    rot = np.array([[np.cos(angle),-np.sin(angle)], [np.sin(angle), np.cos(angle)]])
    xy_out = rot.dot(xy_in)
    return( xy_out )

def secondary_vertex ( row1, row2 ):
    XY1, XY2 = [to_global_coord(row1), to_global_coord(row2)]
    x1 = XY1[0]
    y1 = XY1[1]
    x2 = XY2[0]
    y2 = XY2[1]
    
    px1 = row1["px"]
    px2 = row2["px"]
    py1 = row1["py"]
    py2 = row2["py"]
    pz1 = row1["pz"]
    pz2 = row2["pz"]
    
    m1 = py1/px1
    m2 = py2/px2
    q1 = y1 - m1*x1
    q2 = y2 - m2*x2
    x_SV = ( q2 - q1 )/( m1 - m2)
    y_SV = y1 + m1*(x_SV - x1)
   
    z1_track = pz1/px1 * (x_SV - x1) + row1["fZ"]
    z2_track = pz2/px2 * (x_SV - x2) + row2["fZ"]
    z_SV = (z1_track + z2_track)/2
    return ([x_SV, y_SV, z_SV])

In [5]:
# Load T-Trees from directories
names_track_extr = file.keys(filter_name=r"*O2filtertrackextr")
names_track      = file.keys(filter_name=r"*O2filtertrack")
names_coll       = file.keys(filter_name=r"*O2collision_001")

# number of subsets to create: this is due to memory problems when doing the combinatorial
N_SPLITS = 10
subsets = np.array_split(range(0,len(names_coll )),N_SPLITS)

collision_offset = 0          # variable to correct the index of the collision (because it starts from 0 at each new TTree)

# Particle masses in GeV
m_K = 0.493677
m_pi = 0.139570
m_d0 = 1.86484
LOWER_MASS = 0.8*m_d0
UPPER_MASS = 1.2*m_d0

start = datetime.now()

for J in range (len(subsets)):
    time_J = datetime.now()
    print(f"Inizio calcolo sul chunk {J} alle {time_J.strftime('%H:%M:%S')}")
    
    list_of_df = []               # add the dataframes in a list (we will concat them later)
    
    for i in subsets[J]:
    
        # Read collision tree
        df_coll = file[ names_coll[i] ].arrays(["fPosX", "fPosY", "fPosZ"], library="pd")   # I take the fPosZ column as a DataFrame
        # list_of_collision_df.append(df_coll)               # add the dataframe in a list (we will concat them later)
    
        # # Read track and trackextr using boolean mask for track and trackextr: NEW: removed TOF/TPC for proton
        df_trackextr = file[ names_track_extr[i] ].arrays(["fPt", "fEta", "fCharge", "fDcaXY",
             "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTOFpi", "fNsigmaTOFka"], library="pd" )
        df_track = file[ names_track[i] ].arrays(["fIndexCollisions", "fAlpha", "fX", "fY", "fZ"], library="pd")
        # merge all rows
        df_track = pd.merge(left=df_track, right=df_coll, how="inner", left_on = "fIndexCollisions", right_index=True)
        df_trackextr = pd.merge(left=df_trackextr, right=df_track, how='inner', left_index=True, right_index=True)
        
        # NEW : Assign particle nature hypothesis based on the chosen cuts
        mask_both = df_trackextr.eval(f"({Ka_hyp} & {Pi_hyp})")
        mask_Ka = df_trackextr.eval(Ka_hyp)
        mask_Pi = df_trackextr.eval(Pi_hyp)
        # From the documentation of numpy select:
        # numpy.select(condlist, choicelist, default=0) When multiple conditions are satisfied, the first one encountered in condlist is used.
        df_trackextr["Hyp"] = np.select([mask_both, mask_Ka, mask_Pi], ["Both", "Kaon", "Pion"], default="Bkg")
        mask = mask_Ka | mask_Pi        # create boolean mask
        # Apply the SAME filter to df_trackextr and df_track to keep them aligned (and keep only useful columns):
        df_trackextr = df_trackextr.loc[mask, ["fIndexCollisions","fAlpha", "fX", "fY", "fZ","fPt", "fEta", "fCharge", "fDcaXY",
                                               "Hyp", "fPosX", "fPosY", "fPosZ", "fNsigmaTPCpi", "fNsigmaTPCka", "fNsigmaTOFpi", "fNsigmaTOFka"] ]
    
        # the rows where the fIndexCollision is negative have been excluded thanks to the merge on the index of collision, which can only be
        # non-negative. So, we keep only those with |fPosZ| < 10
        valid = ( df_trackextr["fPosZ"].abs() < 10 ) & ( df_trackextr["fEta"].abs() < 0.8  )
        df_trackextr = df_trackextr[valid].reset_index(drop=True)
 
        df_trackextr["fIndexCollisions"] += collision_offset   # Fix local fIndexCollisions → global index 
    
        # save results:
        # # ALTERNATIVE 2:
        list_of_df.append( df_trackextr )                  # add the dataframe in a list (we will concat them later)
        
        # Update offset for next loop
        collision_offset += len(df_coll)     
    
        # let's free the memory RAM of unused dataframes:
        del df_trackextr
        del df_track
        gc.collect()
    
    
    # # UNCOMMENT FOR ALTERNATIVE 2:
    df = pd.concat(list_of_df, ignore_index=True)

    # Keep only those whose charge is 1 or -1
    df = df[df["fCharge"].isin([-1,1])]
    
    # Merge everything in the total dataframe
    N = len(df)

    # moment columns:
    df["px"] = df["fPt"] * np.cos(df["fAlpha"])
    df["py"] = df["fPt"] * np.sin(df["fAlpha"])
    df["pz"] = df["fPt"] * np.sinh(df["fEta"])
    
    # energy column (differentiating pions and kaons):
    mass = np.where(df["fCharge"] > 0, m_pi, m_K)
    anti_mass = np.where(df["fCharge"] > 0, m_K, m_pi)
    df["Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + mass**2)
    df["Anti-Ene"] = np.sqrt((df["fPt"] * np.cosh(df["fEta"]))**2 + anti_mass**2)
    
    # Debug
    print("The starting dataframe has", len(df), "rows and ", len(df.columns), "columns.")
    memory = df.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The starting dataframe occupies {memory:.2f} MB")
    # df.head()

    # ALTERNATIVE 3:
    # let's initialize some lists, then we will create a dataframe
    collision_indices = []
    track1_indices = []
    track2_indices = []
    dcaXY_products = []
    inv_masses = []
    anti_masses = []
    pt_totals = []
    pz_totals = []
    SV_X = []
    SV_Y = []
    SV_Z = []
    decay_lengths = []
    cos_pointings = []
    mother = []
    positive_tof_ka = []
    positive_tof_pi = []
    negative_tof_ka = []
    negative_tof_pi = []
    positive_tpc_ka = []
    positive_tpc_pi = []
    negative_tpc_ka = []
    negative_tpc_pi = []
    positive_pt = []
    negative_pt = []
    
    counting=0 # debug variable
    
    # let's divide the dataframe for positive and negative charged
    df_pos = df[ df['fCharge']>0 ]
    df_neg = df[ df['fCharge']<0 ]
    
    # Iterate over each collision group
    for collision_idx in (df_neg['fIndexCollisions'].unique()):
        group_pos = df_pos[ df_pos['fIndexCollisions'] == collision_idx ]
        group_neg = df_neg[ df_neg['fIndexCollisions'] == collision_idx ]
    
        # Only collisions with at least a pair
        if len(group_pos) < 1:   continue
    
        # Reset index of the group to 0..N-1 and move original index in new column 'orig_index'
        group_pos = group_pos.reset_index().rename(columns={'index': 'orig_index'})
        group_neg = group_neg.reset_index().rename(columns={'index': 'orig_index'})
    
        # let's crate indexes for all possible pairs:
        combinat = itertools.product( range(len(group_neg)), range(len(group_pos)) )
    
        # Iterate over all unique pairs of tracks
        for combo in combinat:
            row_neg = group_neg.iloc[combo[0]]
            row_pos = group_pos.iloc[combo[1]]
    
            product_dcaXY = row_neg['fDcaXY'] * row_pos['fDcaXY']
            
            # INVARIANT MASS calculation
            pt1, pt2 = row_neg['fPt'], row_pos['fPt']
   
            # exact formula:
            E1 = row_neg['Ene']
            E2 = row_pos['Ene']
            px1 = row_neg["px"]
            py1 = row_neg["py"]
            pz1 = row_neg["pz"]
            px2 = row_pos["px"]
            py2 = row_pos["py"]
            pz2 = row_pos["pz"]
            inv_mass = np.sqrt( (E1+E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )

            anti_E1 = row_neg['Anti-Ene']
            anti_E2 = row_pos['Anti-Ene']
            anti_inv_mass = np.sqrt( (anti_E1+anti_E2)**2 - (px1+px2)**2 - (py1+py2)**2 - (pz1+pz2)**2 )

            # NEW: Check if at least 1 hypothesis is within the specified interval; if not, skip this entry
            is_within_mass_window = (  ( (inv_mass > LOWER_MASS) & (inv_mass < UPPER_MASS) ) |
                                       ( (anti_inv_mass > LOWER_MASS) & (anti_inv_mass < UPPER_MASS) ) 
                                    )
            if ( is_within_mass_window == False ): continue

            mother_hyp = ""

            # NEW: Assign mass of the mother based on the hypothesis on the daughters
            # inv_mass has been calculated under the assumption that the positive particle is the Pion, so the mother is D0
            if (  ( (row_neg["Hyp"] in ["Kaon", "Both"]) & (row_pos["Hyp"] == "Pion") )  | \
                  ( (row_neg["Hyp"] == "Kaon" ) & (row_pos["Hyp"] in ["Pion", "Both"] ) )  ):
                mother_hyp = "D0"

            elif( ( (row_neg["Hyp"] == "Pion" ) & (row_pos["Hyp"] in ["Kaon", "Both"] ) )| \
                  ( (row_neg["Hyp"] in ["Pion", "Both"] ) & (row_pos["Hyp"]=="Kaon" ) ) ):
                mother_hyp = "Anti-D0"

            else:
                mother_hyp = "Undecided"
    
    
            # total transverse momentum of the D0 candidate (used later for sliced plots)
            pt_total = np.sqrt((px1 + px2)**2 + (py1 + py2)**2)
    
            # secondary vertex
            SV_coords = np.array( secondary_vertex(row_neg, row_pos) )
    
            # decay length: distance between PV and SV
            PV_coords = np.array( [row_pos["fPosX"], row_pos["fPosY"], row_pos["fPosZ"]] )
            decay_lengths.append( np.linalg.norm(SV_coords - PV_coords ) )
    
            # cosine of pointing angle: the latter is the angle between the direction of the mother particle and the line connecting PV and SV
            mother_direction = [px1+px2, py1+py2, pz1+pz2]
            flight_line = SV_coords - PV_coords
            cos_pointings.append ( np.dot(mother_direction, flight_line)/(np.linalg.norm(mother_direction)*np.linalg.norm(flight_line)) )

            #save single track variables to perform cuts in the future
            p_pt = row_pos['fPt']
            n_pt = row_neg['fPt']
            p_tof_ka = row_pos['fNsigmaTOFka']
            p_tof_pi = row_pos['fNsigmaTOFpi']
            n_tof_ka = row_neg['fNsigmaTOFka']
            n_tof_pi = row_neg['fNsigmaTOFpi']
            p_tpc_ka = row_pos['fNsigmaTPCka']
            p_tpc_pi = row_pos['fNsigmaTPCpi']
            n_tpc_ka = row_neg['fNsigmaTPCka']
            n_tpc_pi = row_neg['fNsigmaTPCpi']

            # let's add the found pairs to the lists
            collision_indices.append(int(row_neg['fIndexCollisions']))
            dcaXY_products.append(product_dcaXY)
            inv_masses.append(inv_mass)
            anti_masses.append(anti_inv_mass)
            pt_totals.append(pt_total)
            pz_totals.append(pz1+pz2)
            mother.append(mother_hyp)
            positive_tof_ka.append(p_tof_ka)
            positive_tof_pi.append(p_tof_pi)
            negative_tof_ka.append(n_tof_ka)
            negative_tof_pi.append(n_tof_pi)
            positive_tpc_ka.append(p_tpc_ka)
            positive_tpc_pi.append(p_tpc_pi)
            negative_tpc_ka.append(n_tpc_ka)
            negative_tpc_pi.append(p_tpc_pi)
            positive_pt.append(p_pt)
            negative_pt.append(n_pt)
    
        # # let's free the memory RAM of unused dataframes:
        # # PROBLEM: THIS IS VERY SLOW!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
        # del group
        # gc.collect()
    
        # debug:
        counting += 1
        if counting % 50000 ==0: print(counting, end=' ')
    
    
    # create a dataframe with the result:
    df_pairs = pd.DataFrame({
        'collision_index': np.array(collision_indices, dtype="uint32"),
        'dcaXY_product': np.array(dcaXY_products, dtype="float32"),
        'inv_mass': np.array(inv_masses, dtype="float32"),
        'anti_mass': np.array(anti_masses, dtype="float32"),
        'pt': np.array(pt_totals, dtype="float32"),
        'pz': np.array(pz_totals, dtype="float32"),
        'decay_length': np.array(decay_lengths, dtype="float32"),
        'cos_pointing': np.array(cos_pointings, dtype="float32"),
        'particle': mother,
        'pos_tof_ka': np.array(positive_tof_ka, dtype="float32"),
        'pos_tof_pi': np.array(positive_tof_pi, dtype="float32"),
        'neg_tof_ka': np.array(negative_tof_ka, dtype="float32"),
        'neg_tof_pi': np.array(negative_tof_pi, dtype="float32"),
        'pos_tpc_ka': np.array(positive_tpc_ka, dtype="float32"),
        'pos_tpc_pi': np.array(positive_tpc_pi, dtype="float32"),
        'neg_tpc_ka': np.array(negative_tpc_ka, dtype="float32"),
        'neg_tpc_pi': np.array(negative_tpc_pi, dtype="float32"),
        'pos_pt': np.array(positive_pt, dtype="float32"),
        'neg_pt': np.array(negative_pt, dtype="float32"),
    })

    # Debug
    print("The final dataframe has", len(df_pairs), "rows")
    memory = df_pairs.memory_usage(deep=True).sum() / (1024 ** 2)
    print(f"The dataframe occupy {memory:.2f} MB")
    display( pd.concat([df_pairs.head(2),df_pairs.tail(3)]) )

    save_name = "pairs_" + "_".join([which_chunk, which_number, str(J)])
    df_pairs.to_pickle("/home/mattia/Desktop/PhysicsOfData/LCPB_Project/Big_Combinatorial/" + save_name + "_v2.pkl")
    
    # free memory from the dataframes used for this subset
    del df
    del df_pairs
    del collision_indices 
    del dcaXY_products
    del inv_masses
    del pt_totals
    del pz_totals
    del decay_lengths
    del cos_pointings
    gc.collect()

    print(f"Iteration {J+1} out of {N_SPLITS} done!")
    print(f"{save_name}_v2.pkl created.")
    time_J_end = datetime.now()
    print(f"Fine del calcolo sul chunk {J} alle {time_J_end.strftime('%H:%M:%S')}")

end = datetime.now()
duration = end - start
print(f"Il calcolo del combinatorio ha impiegato in totale: ", duration )

Inizio calcolo sul chunk 0 alle 10:08:51
The starting dataframe has 290653 rows and  22 columns.
The starting dataframe occupies 40.19 MB
The final dataframe has 173567 rows
The dataframe occupy 21.05 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle,pos_tof_ka,pos_tof_pi,neg_tof_ka,neg_tof_pi,pos_tpc_ka,pos_tpc_pi,neg_tpc_ka,neg_tpc_pi,pos_pt,neg_pt
0,0,-3.887406e-07,2.024925,2.004279,0.562010,0.217750,0.002837,0.119625,Undecided,-999.000000,-999.000000,-999.0,-999.0,0.336417,0.494370,-1.040863,0.494370,1.062110,0.884602
1,0,-1.776956e-05,1.630082,1.723747,1.006076,0.707632,0.014159,-0.858506,Anti-D0,2.731402,47.058315,-999.0,-999.0,1.979263,9.544146,0.791008,9.544146,0.537510,1.256379
173564,62378,4.459377e-06,2.011489,1.971816,0.364719,-0.906929,0.053541,-0.348906,Anti-D0,2.052955,17.505087,-999.0,-999.0,0.741958,0.198015,-0.951315,0.198015,1.121485,0.767301
173565,62378,-2.193734e-06,2.307482,2.236472,0.790156,-1.256833,0.003415,-0.432140,Undecided,-999.000000,-999.000000,-999.0,-999.0,0.521797,-1.078882,-0.951315,-1.078882,1.511223,0.767301
173566,62381,-8.943851e-06,1.842400,1.563067,2.372743,1.522367,0.011118,0.309042,Anti-D0,-999.000000,-999.000000,-999.0,-999.0,2.799455,0.261475,-6.222051,0.261475,2.090272,0.471232


Iteration 1 out of 3 done!
pairs_Chunk0950_001_007_0_v2.pkl created.
Fine del calcolo sul chunk 0 alle 10:11:50
Inizio calcolo sul chunk 1 alle 10:11:50
The starting dataframe has 292494 rows and  22 columns.
The starting dataframe occupies 40.45 MB
The final dataframe has 177825 rows
The dataframe occupy 21.57 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle,pos_tof_ka,pos_tof_pi,neg_tof_ka,neg_tof_pi,pos_tpc_ka,pos_tpc_pi,neg_tpc_ka,neg_tpc_pi,pos_pt,neg_pt
0,62386,6.408837e-06,1.572182,1.580346,0.967084,0.069280,0.008206,0.415631,Undecided,-999.00000,-999.000000,-999.0,-999.0,-2.297795,-0.540116,-2.161057,-0.540116,0.645943,0.662701
1,62387,4.870554e-07,2.173418,2.311768,1.462494,-0.189875,0.275542,-0.966174,D0,-31.75691,2.642995,-999.0,-999.0,-4.281634,1.647770,1.853538,1.647770,0.532743,1.995227
177822,125022,-3.977024e-06,1.572266,1.628645,0.279431,-0.807337,0.016382,-0.490644,D0,-27.82972,0.486005,-999.0,-999.0,-3.651053,0.328606,-1.850803,0.328606,0.626006,0.766444
177823,125022,5.309686e-06,1.515639,1.557259,0.647694,-0.910794,0.011663,0.582421,Undecided,-999.00000,-999.000000,-999.0,-999.0,-0.540596,2.506171,-1.850803,2.506171,0.696739,0.766444
177824,125022,6.768405e-06,1.583819,1.397978,1.876467,0.143709,0.007391,0.479173,Undecided,-999.00000,-999.000000,-999.0,-999.0,4.331104,2.698508,-5.322775,2.698508,1.734690,0.466070


Iteration 2 out of 3 done!
pairs_Chunk0950_001_007_1_v2.pkl created.
Fine del calcolo sul chunk 1 alle 10:14:53
Inizio calcolo sul chunk 2 alle 10:14:53
The starting dataframe has 293954 rows and  22 columns.
The starting dataframe occupies 40.65 MB
The final dataframe has 178338 rows
The dataframe occupy 21.63 MB


,collision_index,dcaXY_product,inv_mass,anti_mass,pt,pz,decay_length,cos_pointing,particle,pos_tof_ka,pos_tof_pi,neg_tof_ka,neg_tof_pi,pos_tpc_ka,pos_tpc_pi,neg_tpc_ka,neg_tpc_pi,pos_pt,neg_pt
0,125025,-1.673846e-05,1.560681,1.411413,0.846876,1.103823,0.005950,-0.169573,Undecided,-12.330068,-0.40812,-37.457176,1.992431,1.444435,0.446591,-6.008806,0.446591,1.104470,0.412002
1,125025,8.673495e-06,1.953268,1.852170,0.635491,0.610814,0.034427,0.946561,Undecided,-12.330068,-0.40812,-999.000000,-999.000000,1.444435,0.446591,-6.695238,0.446591,1.104470,0.506859
178335,187428,6.208118e-06,2.121660,2.117428,1.318938,-0.120833,0.005220,0.197411,Undecided,-999.000000,-999.00000,-999.000000,-999.000000,0.297579,-0.156485,-0.467412,-0.156485,0.926000,0.954526
178336,187428,-6.499863e-06,1.679689,1.753841,0.354555,0.825447,0.059229,-0.576323,D0,-999.000000,-999.00000,-999.000000,-999.000000,-4.855767,-0.765929,-0.467412,-0.765929,0.600937,0.954526
178337,187429,-6.172655e-08,2.224842,2.265366,0.298633,-1.020032,0.008704,0.474082,Undecided,-999.000000,-999.00000,-999.000000,-999.000000,-1.506331,-0.513226,0.227289,-0.513226,0.928782,1.099519


Iteration 3 out of 3 done!
pairs_Chunk0950_001_007_2_v2.pkl created.
Fine del calcolo sul chunk 2 alle 10:17:57
Il calcolo del combinatorio ha impiegato in totale:  0:09:05.466829
